## KYHPS 데이터 전처리 (Data Preprocessing)
: 데이터 구조 확인(05번)에서 도출된 문제점을 기반으로
4가지 전처리 전략을 구현·비교.

- 확인된 문제점 → 대응 전략 매핑:
  1. 탈락 결측 (21.1%)           → 전략 1: Complete Case / 전략 3: LOCF / 전략 4: MICE
  2. 구조적 결측 (skip logic)     → 공통: 구조적 결측 마스킹 후 전략별 처리
  3. 코딩 역전 (1=예/2=아니오)   → 공통: 이분형 변수 0/1 재코딩
  4. 음주 이력 역전 (45.7%)       → 전략 2: 보수적 재코딩 / 전략 4: MICE 대입
  5. 복합표본 가중치              → 공통: 가중치 컬럼 보존 (전처리에서 제거 금지)
  6. 파별 신규 추가 변수          → 전략 1·3·4: 전 파 공통 변수로 제한

In [8]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer   # MICE용
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
import warnings
warnings.filterwarnings('ignore')

In [9]:
# ─────────────────────────────────────────────
# 0. 데이터 로딩 & 공통 준비
# ─────────────────────────────────────────────
DATA_PATH = r"C:\Users\jiwoo\OneDrive\Desktop\나\3학년\제1회 청소년건강패널조사\2019-23_student_merged_wide_format.csv"
df_raw = pd.read_csv(DATA_PATH, low_memory=False, encoding='cp949')

WAVES = ['w1', 'w2', 'w3', 'w4', 'w5']
WAVE_YEARS = {'w1': 2019, 'w2': 2020, 'w3': 2021, 'w4': 2022, 'w5': 2023}

# 응답 지표 생성
for w in WAVES:
    df_raw[f'responded_{w}'] = df_raw[f'종단면 가중치_{w}'].notna().astype(int)
resp_cols = [f'responded_{w}' for w in WAVES]
df_raw['waves_completed'] = df_raw[resp_cols].sum(axis=1)
df_raw['is_completer'] = (df_raw['waves_completed'] == 5).astype(int)

# 전 파 공통 분석 변수 목록 (가중치·설계변수·응답지표 제외)
EXCLUDE_KEYWORDS = ['가중치', '보정계수', 'responded_', 'is_completer', 'waves_completed',
                    '설문번호', '학교담당자', '시도코드', '군집', '층화']
wave_col_bases = {}
for w in WAVES:
    wave_col_bases[w] = set(
        c.replace(f'_{w}', '') for c in df_raw.columns
        if c.endswith(f'_{w}') and not any(kw in c for kw in EXCLUDE_KEYWORDS)
    )
COMMON_BASE_VARS = sorted(set.intersection(*wave_col_bases.values()))
# 분석용 변수 전체 목록 (파별 확장)
ANALYSIS_VARS = [f'{base}_{w}' for base in COMMON_BASE_VARS for w in WAVES
                 if f'{base}_{w}' in df_raw.columns]
# 보존 변수 (ID + 설계변수 + 가중치)
DESIGN_VARS = ['설문번호'] + \
              [c for c in df_raw.columns if any(kw in c for kw in ['가중치', '보정계수', '군집', '층화'])]
DESIGN_VARS = [c for c in DESIGN_VARS if c in df_raw.columns]

print(f"전 파 공통 분석 변수: {len(COMMON_BASE_VARS)}개 기저 × 5파 = {len(ANALYSIS_VARS)}개 컬럼")

전 파 공통 분석 변수: 161개 기저 × 5파 = 805개 컬럼


In [10]:
# ─────────────────────────────────────────────
# [공통 전처리] 모든 전략에 공통 적용
#   C1. 이분형 변수 재코딩: 1→1, 2→0 (예/아니오 구조)
#   C2. 구조적 결측 표시: skip_logic 컬럼 생성
#   C3. 음주 이력 역전 탐지 플래그
# ─────────────────────────────────────────────
def apply_common_preprocessing(df: pd.DataFrame) -> pd.DataFrame:
    """
    모든 전략 공통 전처리.
    원본 df_raw를 직접 수정하지 않도록 복사본에 적용.
    """
    d = df.copy()

    # C1. 이분형 변수 재코딩 (1=예→1, 2=아니오→0)
    #     unique 값이 {1,2} 또는 {1.0,2.0} 인 컬럼에만 적용
    binary_cols = []
    for col in d.select_dtypes(include=np.number).columns:
        vals = set(d[col].dropna().unique())
        if vals <= {1, 2, 1.0, 2.0}:
            binary_cols.append(col)

    recode_map = {1: 1, 2: 0, 1.0: 1, 2.0: 0}
    d[binary_cols] = d[binary_cols].replace(recode_map)
    print(f"  [C1] 이분형 재코딩 완료: {len(binary_cols)}개 컬럼 (1=예, 0=아니오)")

    # C2. 구조적 결측 식별 (예: 음주 잔 기준은 모금 기준 '있음'(→1 재코딩 후)인 경우만 유효)
    for w in WAVES:
        sip = f'평생 음주 경험(모금 기준)_{w}'
        glass = f'평생 음주 경험(잔 기준)_{w}'
        if sip in d.columns and glass in d.columns:
            # 모금=0(아니오)이면 잔 기준은 구조적으로 해당 없음 → NaN 유지 (skip logic)
            struct_mask = (d[sip] == 0) & d[glass].isna()
            d[f'skip_drink_glass_{w}'] = struct_mask.astype(int)
    skip_cols = [c for c in d.columns if c.startswith('skip_')]
    print(f"  [C2] 구조적 결측 플래그 생성: {len(skip_cols)}개 컬럼")

    # C3. 음주 이력 역전 탐지 (잔 기준)
    drink_vars = [f'평생 음주 경험(잔 기준)_{w}' for w in WAVES if f'평생 음주 경험(잔 기준)_{w}' in d.columns]
    reversal_count = pd.Series(0, index=d.index)
    for i in range(1, len(drink_vars)):
        prev = d[drink_vars[i-1]]
        curr = d[drink_vars[i]]
        valid = prev.notna() & curr.notna()
        rev = ((prev == 1) & (curr == 0) & valid).astype(int)
        reversal_count += rev
    d['drink_reversal_count'] = reversal_count
    d['has_reversal'] = (reversal_count > 0).astype(int)
    print(f"  [C3] 역전 탐지 완료: 역전 보유 {d['has_reversal'].sum():,}명")

    return d


print("=" * 65)
print("[ 공통 전처리 실행 ]")
print("=" * 65)
df_common = apply_common_preprocessing(df_raw)

[ 공통 전처리 실행 ]
  [C1] 이분형 재코딩 완료: 245개 컬럼 (1=예, 0=아니오)
  [C2] 구조적 결측 플래그 생성: 5개 컬럼
  [C3] 역전 탐지 완료: 역전 보유 427명


In [11]:
# ═══════════════════════════════════════════════════════════════
# 전략 1: Complete Case Analysis (CCA)
# 기준: 5개 파 모두 응답한 3,986명만 보유
# 장점: 시간 흐름 일관성 보장, 복합표본 가중치 그대로 적용 가능
# 단점: 탈락 편향 수용 (음주 경험 기저치 유의차 p=.034)
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("[ 전략 1: Complete Case Analysis (CCA) ]")
print("=" * 65)

df_cca = df_common[df_common['is_completer'] == 1].copy()
df_cca = df_cca[DESIGN_VARS + ANALYSIS_VARS +
                [c for c in df_cca.columns if c.startswith('skip_') or
                 c in ['drink_reversal_count', 'has_reversal',
                        'waves_completed', 'is_completer']]].copy()

print(f"  포함 피험자: {len(df_cca):,}명 (원본 {len(df_raw):,}명의 {len(df_cca)/len(df_raw)*100:.1f}%)")
print(f"  잔여 결측값: {df_cca[ANALYSIS_VARS].isnull().sum().sum():,}개")
print(f"    ※ 잔여 결측 = 구조적 결측(skip logic) + 완전응답자 내 부분 무응답")
print(f"  사용 권고: 주 분석(Main Analysis). 제한점에 탈락 편향 명시 필수.")

# 전략 1 내 음주 역전 처리: 보수적 재코딩 (forced=1) 유지
print(f"  음주 역전 처리: 보수적 재코딩(있음=1) 유지 — {df_cca['has_reversal'].sum()}명 해당")

out1 = r"C:\Users\jiwoo\OneDrive\Desktop\나\3학년\제1회 청소년건강패널조사\output_cleaning\preprocessed_CCA.csv"
df_cca.to_csv(out1, index=False, encoding='utf-8-sig')
print(f"  저장: {out1}")


[ 전략 1: Complete Case Analysis (CCA) ]
  포함 피험자: 3,986명 (원본 5,051명의 78.9%)
  잔여 결측값: 1,172,543개
    ※ 잔여 결측 = 구조적 결측(skip logic) + 완전응답자 내 부분 무응답
  사용 권고: 주 분석(Main Analysis). 제한점에 탈락 편향 명시 필수.
  음주 역전 처리: 보수적 재코딩(있음=1) 유지 — 392명 해당
  저장: C:\Users\jiwoo\OneDrive\Desktop\나\3학년\제1회 청소년건강패널조사\output_cleaning\preprocessed_CCA.csv


In [12]:
# ═══════════════════════════════════════════════════════════════
# 전략 2: 역전 응답 보수적 재코딩 비교 (LOCF 방식)
# 기준: 탈락자 포함 전체 5,051명 유지
#       직전 파 값을 다음 파로 이월 (Last Observation Carried Forward)
# 장점: 탈락자 포함 → 표본 크기 유지, 계산 단순
# 단점: 회복·개선 추세를 과소추정, 결과 보수 편향
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("[ 전략 2: LOCF (Last Observation Carried Forward) ]")
print("=" * 65)

df_locf = df_common.copy()

# LOCF 적용: 파 순서대로 직전 값 이월 (분석 변수에만)
locf_fill_count = 0
for base in COMMON_BASE_VARS:
    cols_in_order = [f'{base}_{w}' for w in WAVES if f'{base}_{w}' in df_locf.columns]
    if len(cols_in_order) < 2:
        continue
    for i in range(1, len(cols_in_order)):
        prev_col, curr_col = cols_in_order[i-1], cols_in_order[i]
        before = df_locf[curr_col].isna().sum()
        df_locf[curr_col] = df_locf[curr_col].fillna(df_locf[prev_col])
        after = df_locf[curr_col].isna().sum()
        locf_fill_count += (before - after)

print(f"  LOCF 이월로 채워진 셀: {locf_fill_count:,}개")
remaining = df_locf[ANALYSIS_VARS].isnull().sum().sum()
print(f"  LOCF 후 잔여 결측: {remaining:,}개 (W1 결측 또는 구조적 결측)")
print(f"  포함 피험자: {len(df_locf):,}명 (전체 표본 유지)")
print(f"  사용 권고: 민감도 분석(Sensitivity Analysis)용. 주 분석 대체 불가.")

out2 = r"C:\Users\jiwoo\OneDrive\Desktop\나\3학년\제1회 청소년건강패널조사\output_cleaning\preprocessed_LOCF.csv"
df_locf.to_csv(out2, index=False, encoding='utf-8-sig')
print(f"  저장: {out2}")


[ 전략 2: LOCF (Last Observation Carried Forward) ]
  LOCF 이월로 채워진 셀: 344,220개
  LOCF 후 잔여 결측: 1,388,646개 (W1 결측 또는 구조적 결측)
  포함 피험자: 5,051명 (전체 표본 유지)
  사용 권고: 민감도 분석(Sensitivity Analysis)용. 주 분석 대체 불가.
  저장: C:\Users\jiwoo\OneDrive\Desktop\나\3학년\제1회 청소년건강패널조사\output_cleaning\preprocessed_LOCF.csv


In [13]:
# ═══════════════════════════════════════════════════════════════
# 전략 3: 역전 응답 LOCF 재코딩 (음주 이력 전용)
# 기준: CCA 기반 + 역전 응답을 직전 파 값으로 대체
#       (보수적 재코딩과의 차이: 역전이 실제 중단을 반영할 수 있음을 허용)
# 장점: 불합리한 forced=1 보다 자연스러운 추세 반영
# 단점: 일시적 단절 후 재개를 포착 못할 수 있음
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("[ 전략 3: 음주 역전 LOCF 재코딩 (CCA 기반) ]")
print("=" * 65)

df_rev_locf = df_common[df_common['is_completer'] == 1].copy()

drink_vars_ordered = [f'평생 음주 경험(잔 기준)_{w}' for w in WAVES
                      if f'평생 음주 경험(잔 기준)_{w}' in df_rev_locf.columns]

rev_fixed = 0
for i in range(1, len(drink_vars_ordered)):
    prev_col = drink_vars_ordered[i-1]
    curr_col = drink_vars_ordered[i]
    # 역전 탐지: 직전=1(있음), 현재=0(없음) → LOCF로 1 유지
    reversal_mask = (df_rev_locf[prev_col] == 1) & (df_rev_locf[curr_col] == 0)
    df_rev_locf.loc[reversal_mask, curr_col] = 1
    rev_fixed += reversal_mask.sum()

print(f"  역전 → LOCF 수정 건수: {rev_fixed:,}건")
remaining_rev = (df_rev_locf[drink_vars_ordered]
                 .apply(lambda col: ((col.shift(1) == 1) & (col == 0)).sum()).sum())
print(f"  수정 후 잔여 역전: {remaining_rev}건")
print(f"  사용 권고: 전략 1과 비교하는 민감도 분석 (음주 유병률 추이 비교)")

out3 = r"C:\Users\jiwoo\OneDrive\Desktop\나\3학년\제1회 청소년건강패널조사\output_cleaning\preprocessed_RevLOCF.csv"
df_rev_locf.to_csv(out3, index=False, encoding='utf-8-sig')
print(f"  저장: {out3}")


[ 전략 3: 음주 역전 LOCF 재코딩 (CCA 기반) ]
  역전 → LOCF 수정 건수: 625건
  수정 후 잔여 역전: 748건
  사용 권고: 전략 1과 비교하는 민감도 분석 (음주 유병률 추이 비교)
  저장: C:\Users\jiwoo\OneDrive\Desktop\나\3학년\제1회 청소년건강패널조사\output_cleaning\preprocessed_RevLOCF.csv


In [14]:
# ═══════════════════════════════════════════════════════════════
# 전략 4: MICE (Multiple Imputation by Chained Equations)
# 기준: CCA 내 구조적 결측 제외한 분석 변수에 다중 대입
#       sklearn IterativeImputer(BayesianRidge) 사용
# 장점: MAR 가정 하 비편향 추정, 불확실성 반영
# 단점: 계산 비용 높음, 복합표본 설계와의 통합 복잡
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("[ 전략 4: MICE (다중 대입법 — 1회 대입 예시) ]")
print("=" * 65)

# MICE는 CCA 대상 내 부분 결측만 대입 (탈락 결측 제외)
df_mice_base = df_common[df_common['is_completer'] == 1].copy()

# MICE 적용 대상: 부분 결측이 있는 분석 변수 (skip logic 제외)
skip_bases = ['평생 음주 경험(잔 기준)']   # 구조적 결측 변수 기저명
mice_target_vars = [v for v in ANALYSIS_VARS
                    if v in df_mice_base.columns
                    and not any(sb in v for sb in skip_bases)
                    and df_mice_base[v].isnull().mean() > 0
                    and df_mice_base[v].isnull().mean() < 1.0]

print(f"  MICE 대입 대상 변수: {len(mice_target_vars)}개")
print(f"  대상 내 결측 셀: {df_mice_base[mice_target_vars].isnull().sum().sum():,}개")

# 1회 대입 (실제 분석에서는 m=5~20 반복 권장)
imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=10,
    random_state=42,
    min_value=0,    # 재코딩 후 최솟값
    max_value=None
)
imputed_arr = imputer.fit_transform(df_mice_base[mice_target_vars])
df_mice = df_mice_base.copy()
df_mice[mice_target_vars] = np.round(imputed_arr).astype(float)  # 범주형이므로 반올림

# 이분형 변수 클리핑 (0~1)
binary_mice = [v for v in mice_target_vars if df_mice[v].nunique() <= 3]
df_mice[binary_mice] = df_mice[binary_mice].clip(0, 1)

remaining_mice = df_mice[mice_target_vars].isnull().sum().sum()
print(f"  MICE 후 잔여 결측: {remaining_mice}개")
print(f"  ※ 실제 논문 적용 시 m=10 반복 후 Rubin's Rules로 추정치 결합 권장")
print(f"  사용 권고: 탈락 편향 민감도 분석, IPW 결과와 비교")

out4 = r"C:\Users\jiwoo\OneDrive\Desktop\나\3학년\제1회 청소년건강패널조사\output_cleaning\preprocessed_MICE_imp1.csv"
df_mice.to_csv(out4, index=False, encoding='utf-8-sig')
print(f"  저장: {out4}")


[ 전략 4: MICE (다중 대입법 — 1회 대입 예시) ]
  MICE 대입 대상 변수: 361개
  대상 내 결측 셀: 1,065,073개
  MICE 후 잔여 결측: 0개
  ※ 실제 논문 적용 시 m=10 반복 후 Rubin's Rules로 추정치 결합 권장
  사용 권고: 탈락 편향 민감도 분석, IPW 결과와 비교
  저장: C:\Users\jiwoo\OneDrive\Desktop\나\3학년\제1회 청소년건강패널조사\output_cleaning\preprocessed_MICE_imp1.csv


In [15]:
# ─────────────────────────────────────────────
# 전략별 결측 처리 결과 비교 요약
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("[ 전략별 비교 요약 ]")
print("=" * 65)

drink_base = '평생 음주 경험(잔 기준)'
summary_data = [
    {
        '전략': '1. CCA',
        'N': len(df_cca),
        '잔여결측': df_cca[ANALYSIS_VARS].isnull().sum().sum(),
        f'음주유병률_w1(%)': (df_cca.get(f'{drink_base}_w1', pd.Series([np.nan])) == 1).mean() * 100,
        f'음주유병률_w5(%)': (df_cca.get(f'{drink_base}_w5', pd.Series([np.nan])) == 1).mean() * 100,
        '탈락편향보정': '❌',
        '역전보정': '보수적(forced=1)',
    },
    {
        '전략': '2. LOCF',
        'N': len(df_locf),
        '잔여결측': df_locf[ANALYSIS_VARS].isnull().sum().sum(),
        f'음주유병률_w1(%)': (df_locf.get(f'{drink_base}_w1', pd.Series([np.nan])) == 1).mean() * 100,
        f'음주유병률_w5(%)': (df_locf.get(f'{drink_base}_w5', pd.Series([np.nan])) == 1).mean() * 100,
        '탈락편향보정': '△(이월)',
        '역전보정': 'LOCF 이월',
    },
    {
        '전략': '3. RevLOCF',
        'N': len(df_rev_locf),
        '잔여결측': df_rev_locf[ANALYSIS_VARS].isnull().sum().sum(),
        f'음주유병률_w1(%)': (df_rev_locf.get(f'{drink_base}_w1', pd.Series([np.nan])) == 1).mean() * 100,
        f'음주유병률_w5(%)': (df_rev_locf.get(f'{drink_base}_w5', pd.Series([np.nan])) == 1).mean() * 100,
        '탈락편향보정': '❌',
        '역전보정': 'LOCF(역전만)',
    },
    {
        '전략': '4. MICE',
        'N': len(df_mice),
        '잔여결측': df_mice[mice_target_vars].isnull().sum().sum(),
        f'음주유병률_w1(%)': (df_mice.get(f'{drink_base}_w1', pd.Series([np.nan])) == 1).mean() * 100,
        f'음주유병률_w5(%)': (df_mice.get(f'{drink_base}_w5', pd.Series([np.nan])) == 1).mean() * 100,
        '탈락편향보정': '✅(MAR)',
        '역전보정': 'MICE 대입',
    },
]
summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

print("""
권장 사용 전략:
  주 분석      → 전략 1 (CCA) + 복합표본 가중치 적용
  민감도 분석  → 전략 2(LOCF) vs 전략 3(RevLOCF) 비교
  편향 보정    → 전략 4(MICE) 또는 IPW 적용
  제한점 기술  → 방법 선택의 근거와 방향성을 논문 Limitations에 명시
""")


[ 전략별 비교 요약 ]
        전략    N    잔여결측  음주유병률_w1(%)  음주유병률_w5(%) 탈락편향보정          역전보정
    1. CCA 3986 1172543     7.651781    25.740090      ❌ 보수적(forced=1)
   2. LOCF 5051 1388646     8.077608    25.777074  △(이월)       LOCF 이월
3. RevLOCF 3986 1172543     7.651781    30.858003      ❌     LOCF(역전만)
   4. MICE 3986       0     7.651781    25.740090 ✅(MAR)       MICE 대입

권장 사용 전략:
  주 분석      → 전략 1 (CCA) + 복합표본 가중치 적용
  민감도 분석  → 전략 2(LOCF) vs 전략 3(RevLOCF) 비교
  편향 보정    → 전략 4(MICE) 또는 IPW 적용
  제한점 기술  → 방법 선택의 근거와 방향성을 논문 Limitations에 명시

